## Langsmith Weights + Biases Integration

- Enable langsmith to trace Langchain runs end to end
- Track latency, token usage, and outputs to Weights and Biases
- Log a batch evaluation
- Capture config and environment for reproducibility.

In [12]:
from dotenv import load_dotenv
import os

load_dotenv()

AZURE_OPENAI_KEY = os.getenv("AZURE_OPENAI_KEY")
AZURE_OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT")
AZURE_OPENAI_VERSION = os.getenv("AZURE_OPENAI_VERSION")
LANGCHAIN_API_KEY = os.getenv("LANGCHAIN_API_KEY")
WANDB_API_KEY = os.getenv("WANDB_API_KEY")

In [13]:
import os

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = LANGCHAIN_API_KEY
os.environ["LANGCHAIN_PROJECT"] = "exp"

In [14]:
default_temp = 0.2
model_name = "gpt4o"

In [15]:
from langchain_openai import AzureChatOpenAI

llm = AzureChatOpenAI(
    azure_endpoint= AZURE_OPENAI_ENDPOINT,
    azure_deployment="gpt-4o-mini",
    openai_api_key=AZURE_OPENAI_KEY,
    openai_api_version=AZURE_OPENAI_VERSION,
    temperature=default_temp)

In [16]:
PROJECT_NAME = "exp" #langchain project name
RUN_NAME = "run1" #langchain run name

In [17]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a concise assistant. Keep answers short, correct and well-formatted."),
    ("human", "{question}")
])
chain = prompt | llm | StrOutputParser()

In [18]:
import os
os.environ['WANDB_API_KEY'] = WANDB_API_KEY

In [9]:
#Initialize weight and biases run
import weave
weave.init('soundarya08012000-none/intro-example') 

weave: Logged in as Weights & Biases user: soundarya08012000.
weave: View Weave data at https://wandb.ai/soundarya08012000-none/intro-example/weave


In [21]:
import time
from langchain_community.callbacks import get_openai_callback
@weave.op
def profile_one(question:str):
    start = time.perf_counter()
    with get_openai_callback() as cb:
        output = chain.invoke({"question": question})
        rec = {
            "question": question,
            "output": output,
            "tokens": cb.total_tokens,
            "prompt_tokens": cb.prompt_tokens,
            "completion_tokens": cb.completion_tokens,
            "total_cost": cb.total_cost
        }
    end = time.perf_counter()
    print(f"Time taken for {question}: {end - start}")
    return rec


In [22]:
questions = [
    "What is the capital of France?",
    "Who won the FIFA World Cup in 2018?",
]
for question in questions:
    print(profile_one(question))


weave: 🍩 https://wandb.ai/soundarya08012000-none/intro-example/r/call/019d51c6-f38b-7a4d-a01f-47c0259e740d
weave: 🍩 https://wandb.ai/soundarya08012000-none/intro-example/r/call/019d51c6-fa0d-70a9-9978-98bd1c02f047


Time taken for What is the capital of France?: 1.664642400000048
{'question': 'What is the capital of France?', 'output': 'The capital of France is Paris.', 'tokens': 42, 'prompt_tokens': 34, 'completion_tokens': 8, 'total_cost': 2.6400000000000005e-05}
Time taken for Who won the FIFA World Cup in 2018?: 1.445767899999737
{'question': 'Who won the FIFA World Cup in 2018?', 'output': 'France won the FIFA World Cup in 2018.', 'tokens': 50, 'prompt_tokens': 38, 'completion_tokens': 12, 'total_cost': 3.44e-05}
